# Clinical Plausibility Constraints

Adversarial attacks on image classifiers can perturb pixel values by tiny amounts. The
change is invisible to humans but fools the model. Tabular health data is fundamentally
different. Every feature has a **real-world meaning** and not all perturbations are
physically or clinically possible.

An attack that succeeds by setting a patient's Glucose to -40 mg/dL or BMI to 0.0 is
**clinically meaningless**. No real patient could submit those values. If we count such
attacks as successes our robustness analysis is inflated.

**This section defines the constraint layer that wraps every attack.**
No adversarial example will be accepted as valid unless it passes through this layer.

### Threat Model Assumption
We assume a **black-box, clinically-constrained adversary**:
- Has access to model outputs only (not model weights or training data)
- Can only submit values a real human patient could plausibly have
- Cannot change categorical features to impossible values
- Cannot violate logical dependencies between features

This reflects the most realistic attack scenario: a patient manipulating their own
self-reported or measured clinical data to game a risk screener.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field, asdict
from typing import Optional, List

In [2]:
DATA_DIR = '../data/processed/'
OUTPUT_DIR = '../output/'

diabetes_df = pd.read_csv(os.path.join(DATA_DIR, 'diabetes_processed.csv'))
heart_df    = pd.read_csv(os.path.join(DATA_DIR, 'heart_processed.csv'))
stroke_df   = pd.read_csv(os.path.join(DATA_DIR, 'stroke_processed.csv'))

print(f'  Diabetes : {diabetes_df.shape}')
print(f'  Heart    : {heart_df.shape}')
print(f'  Stroke   : {stroke_df.shape}')

  Diabetes : (768, 9)
  Heart    : (1025, 14)
  Stroke   : (4254, 12)
